## Bivariate VAR Experiments

This section contains the corresponding simulation procedures and forecasting experiments.
Users can either generate new synthetic datasets by selecting the simulation parameters or load existing datasets stored in the Data/VAR folder. The provided datasets correspond to the experiments conducted in the thesis and facilitate the replication of the reported results.

### Import Required Libraries

Run the Required Libraries before executing any simulation or forecasting experiment.

In [ ]:
import os
os.chdir("xxxx")
from model.Base import Base
from model.MSVR import MSVR
from model.utility import (
    create_dataset,
    create_dataset_antes,
    rmse,
    CustomMSVR,
    create_dataset_rez,
    rezago_sig
)

from scikeras.wrappers import KerasRegressor

from scipy.linalg import orth
from scipy.stats import multivariate_normal

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import (
    RandomizedSearchCV,
    TimeSeriesSplit,
    train_test_split
)
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.tsa.vector_ar.vecm import VECM, select_coint_rank

import csv
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

### Option 1: Generate Synthetic Data and Run the forecasting experiment
This option generates a synthetic bivariate VAR(1) process. The user can modify the model parameters, sample size, forecasting horizon, and the number of simulation replications. Before running the simulation, the user can define the following parameters:


| Parameter | Description |
|-----------|-------------|
| t | Length of the generated time series. |
| k | Dimension of the multivariate process. |
| p | Number of lags in the VAR model. |
| h | Forecast horizon (steps ahead). |
| col | Target variable used for forecasting evaluation. |


Example:

t = 1000  # Length of the series \
k = 2     # Dimension of the vector Y \
p = 1     # Number of lags \
h = 1     # Forecast horizon \
col = 2   # Target variable 

In [ ]:
#t = [50, 200, 500, 1000, 5000]
t = [50, 200]
k = 2
p = 1
h = 1
col = 2
hiperparametros_svr = {size: [] for size in t}
vectores_soporte = {size: [] for size in t}
train_RMSE_svr = {size: [] for size in t}
test_RMSE_svr = {size: [] for size in t}
hiperparametros_svr_tscv = {size: [] for size in t}
vectores_soporte_tscv = {size: [] for size in t}
train_RMSE_svr_tscv = {size: [] for size in t}
test_RMSE_svr_tscv = {size: [] for size in t}
train_RMSE_var = {size: [] for size in t}
test_RMSE_var = {size: [] for size in t}
tiempo_var = {size: [] for size in t}
tiempo_msvr = {size: [] for size in t}
tiempo_msvr_tscv = {size: [] for size in t}

for size in t:
      a=size
      for no in range(100):
        print(f"-------------------------Size {a}-------------------------")
        print(f"-------------------------Iteration {no}--------------------------")

        # Generate series
        A = np.array([[0.5, 0.4],
                      [0.1, 0.8]])

        initial = np.random.normal(size=(2,))

        serie = np.zeros((2, size))
        serie[:, :1] = initial[:, np.newaxis]

        for i in range(1, size):

            lag = serie[:, i-1:i]

            serie[:, i] = (
                np.dot(A, lag.flatten())
                + np.random.normal(
                    loc=0.0,
                    scale=1.0,
                    size=2
                )
            )

        series = pd.DataFrame(
            serie.T,
            columns=["Y1", "Y2"]
        )

        # --------------------------------
        # Forecasting experiment
        # --------------------------------
        train_size = int(len(series) * 0.7)
        train, test = series.iloc[:train_size], series.iloc[train_size:]
        test = test.reset_index(drop=True)
        #Partial autocorrelation analysis
        pacf_var1 = pacf(train['Y1'], nlags=16)
        pacf_var2 = pacf(train['Y2'], nlags=16)
        banda= 1.96 / np.sqrt(size)
        rezago_elegido_1 = rezago_sig(pacf_var1, banda)
        rezago_elegido_2 = rezago_sig(pacf_var2, banda)
        enumerated_list = list(enumerate(pacf_var1))
        reversed_enumerated_list = list(reversed(enumerated_list))
        filtered_indices = [i for i, x in reversed_enumerated_list if abs(x) > banda]
        #rez= int(min(rezago_elegido_1, rezago_elegido_2))
        rez=p
        print(f"----------------Lag {rez}--------------")
        # Fit the VAR model
        start_time = time.time()
        model_var = VAR(train)
        results_var = model_var.fit(maxlags=p)
        lag_order = results_var.k_ar
        modelo_var_train = []
        modelo_var_test = []
        #Generate in-sample predictions
        train_pred = results_var.fittedvalues
        # Generate out-of-sample forecasts
        test_pred=[]
        input_data = train.values[-rez:]
        for i in range(len(test)):
            pred = results_var.forecast(y=input_data, steps=h)
            test_pred.append(pred[0])
            input_data = np.vstack([input_data[1:], test.values[i:i+1]])
        test_pred = np.array(test_pred)
        train_rmse_var = np.sqrt(mean_squared_error(train.values[rez:], train_pred))
        test_rmse_var = np.sqrt(mean_squared_error(test.values, test_pred))
        train_RMSE_var[size].append(train_rmse_var)
        test_RMSE_var[size].append(test_rmse_var)
        print("Termine de ajustar modelo VAR")
        end_time = time.time()
        execution_time = end_time - start_time
        tiempo_var[size].append(execution_time)

        # Fit the MSVR model
        start_time = time.time()
        fechas = pd.DataFrame(list(range(len(series))))
        total = pd.concat([fechas,series], axis=1).values
        dim=len(total)
        #Dataset construction
        data=Base(total)
        data= data.base
        #Create the supervised learning dataset
        dataset = create_dataset_rez(data,dim,h,col,rez)
        X, Y = dataset[:, :(0 - h*2)], dataset[:, (0-h*2):]
        #Train-test split
        X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size=0.3, shuffle=False)
        #Feature standardization
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        scaler_X.fit(X_train)
        scaler_y.fit(y_train)
        X_train_nor = scaler_X.transform(X_train)
        X_test_nor = scaler_X.transform(X_test)
        y_train_nor = scaler_y.transform(y_train)
        y_test_nor = scaler_y.transform(y_test)
        pipe = Pipeline([
            ('MSVR', CustomMSVR(kernel='rbf', degree=3, gamma=0, coef0=0.0, tol=0.001, C=1.0, epsilon=0.1))
        ])
        hyperparameters = {
            #'MSVR__kernel': ['poly'],
            'MSVR__kernel': ['poly','rbf','linear'],
            'MSVR__degree': [2,5],
            #'MSVR__degree': [1],
            'MSVR__gamma': [0.5,1],
            'MSVR__coef0': [0.1,0.5,1],
            'MSVR__C': [5,9,11,13],
            'MSVR__epsilon':[1,2],
        }

        #Con tscv 
        tscv=TimeSeriesSplit(n_splits=5)
        bm_tscv = RandomizedSearchCV(pipe, hyperparameters, n_iter=15, scoring='neg_mean_squared_error', cv=tscv, verbose=0, error_score='raise')
        best_model_tscv = bm_tscv.fit(X_train_nor, y_train_nor)
        best_params_tscv = bm_tscv.best_params_
        msvr_tscv = MSVR(kernel=bm_tscv.best_params_.get("MSVR__kernel"), gamma=bm_tscv.best_params_.get("MSVR__gamma"),
                        epsilon=bm_tscv.best_params_.get("MSVR__epsilon"), C=bm_tscv.best_params_.get("MSVR__C"),
                        degree=bm_tscv.best_params_.get("MSVR__degree"), coef0=bm_tscv.best_params_.get("MSVR__coef0"), tol=0.01)
        msvr_tscv.fit(X_train_nor, y_train_nor)
        trainPred_svr_nor_tscv = msvr_tscv.predict(X_train_nor)
        testPred_svr_nor_tscv = msvr_tscv.predict(X_test_nor)
        trainPred_svr_tscv  = pd.DataFrame(scaler_y.inverse_transform(trainPred_svr_nor_tscv))
        testPred_svr_tscv  = pd.DataFrame(scaler_y.inverse_transform(testPred_svr_nor_tscv))
        train_rmse_svr_tscv = rmse(y_train, trainPred_svr_tscv)
        test_rmse_svr_tscv = rmse(y_test, testPred_svr_tscv)
        hiperparametros_svr_tscv[size].append(best_params_tscv)
        vectores_soporte_tscv[size].append(msvr_tscv.NSV)
        train_RMSE_svr_tscv[size].append(train_rmse_svr_tscv)
        test_RMSE_svr_tscv[size].append(test_rmse_svr_tscv)
        end_time = time.time()
        execution_time_tscv = end_time - start_time
        tiempo_msvr_tscv[size].append(end_time - start_time) 

        #Sin tscv 
        bm = RandomizedSearchCV(pipe, hyperparameters, n_iter=15, scoring='neg_mean_squared_error', cv=5, verbose=0, error_score='raise', random_state=42)
        best_model = bm.fit(X_train_nor, y_train_nor)
        best_params = bm.best_params_
        msvr = MSVR(kernel=bm.best_params_.get("MSVR__kernel"), gamma=bm.best_params_.get("MSVR__gamma"),
                        epsilon=bm.best_params_.get("MSVR__epsilon"), C=bm.best_params_.get("MSVR__C"),
                        degree=bm.best_params_.get("MSVR__degree"), coef0=bm.best_params_.get("MSVR__coef0"), tol=0.01)
        msvr.fit(X_train_nor, y_train_nor)
        trainPred_svr_nor = msvr.predict(X_train_nor)
        testPred_svr_nor = msvr.predict(X_test_nor)
        trainPred_svr  = pd.DataFrame(scaler_y.inverse_transform(trainPred_svr_nor))
        testPred_svr  = pd.DataFrame(scaler_y.inverse_transform(testPred_svr_nor))
        train_rmse_svr = rmse(y_train, trainPred_svr)
        #print(train_rmse_svr)
        test_rmse_svr = rmse(y_test, testPred_svr)
        hiperparametros_svr[size].append(best_params)
        vectores_soporte[size].append(msvr.NSV)
        train_RMSE_svr[size].append(train_rmse_svr)
        test_RMSE_svr[size].append(test_rmse_svr)
        end_time = time.time()
        execution_time = end_time - start_time
        tiempo_msvr[size].append(end_time - start_time)

filename = f'results_ VAR_Bivariate.csv'
with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Tamaño', 'Modelo', 'Hiperparametros', 'Vectores_soporte', 'Train RMSE', 'Test RMSE', 'Tiempo'])
        for size in t:
            writer.writerow([f'Tamaño: {size}', '', '', '', '', '', ''])
            for i in range(100):
                writer.writerow(['SVR', hiperparametros_svr[size][i], vectores_soporte[size][i], train_RMSE_svr[size][i], test_RMSE_svr[size][i], tiempo_msvr[size][i]])
            for i in range(100):
                writer.writerow(['SVR_tscv', hiperparametros_svr_tscv[size][i], vectores_soporte_tscv[size][i], train_RMSE_svr_tscv[size][i], test_RMSE_svr_tscv[size][i], 'N/A'])
            for i in range(100):  
                writer.writerow(['VAR ', 'N/A', 'N/A', train_RMSE_var[size][i], test_RMSE_var[size][i], tiempo_var[size][i]])
            

        print(f'Resultados guardados en {filename}')
        
 
 
        

### Option 2: Load Existing Data
Instead of generating new data, users can load datasets stored in the repository under the Data/VAR directory. These datasets correspond to the simulation scenarios used in the thesis and allow direct replication of the reported results.\
\
*Important:* When using an existing dataset, the time series generation step can be skipped. However, the parameters t (series length), k (dimension), p (number of lags), h (forecast horizon), and col (target variable) must still be defined, since they are used by the subsequent forecasting, model fitting, and evaluation procedures.

#### *Optional: Generate and Save Synthetic Datasets* 

You can generate and save the synthetic datasets if you would like to keep a backup of the generated series or reuse them in future experiments.

In [ ]:
t = [50, 200, 500, 1000, 5000]
k = 2
p = 1
h = 1
col = 2


# Create folder if it does not exist
os.makedirs("xxxxx", exist_ok=True)

for size in t:

    filename = f"dataset_size_{size}.xlsx"

    with pd.ExcelWriter(filename, engine="openpyxl") as writer:

        for no in range(100):

            A = np.array([[0.5, 0.4],
                          [0.1, 0.8]])

            initial = np.random.normal(size=(2,))

            serie = np.zeros((2, size))
            serie[:, :1] = initial[:, np.newaxis]

            for i in range(1, size):

                lag = serie[:, i-1:i]

                serie[:, i] = (
                    np.dot(A, lag.flatten())
                    + np.random.normal(
                        loc=0.0,
                        scale=1.0,
                        size=2
                    )
                )

            series = pd.DataFrame(
                serie.T,
                columns=["Y1", "Y2"]
            )

            series.to_excel(
                writer,
                sheet_name=f"Iteration_{no+1}",
                index=False
            )
        series.to_csv(
        filename,
        index=False
    )

    print(f"Saved: {filename}") 


## Run the forecasting model

In [ ]:
t = [50, 200, 500, 1000, 5000]
k = 2
p = 1
h = 1
col = 2
hiperparametros_svr = {size: [] for size in t}
vectores_soporte = {size: [] for size in t}
train_RMSE_svr = {size: [] for size in t}
test_RMSE_svr = {size: [] for size in t}
hiperparametros_svr_tscv = {size: [] for size in t}
vectores_soporte_tscv = {size: [] for size in t}
train_RMSE_svr_tscv = {size: [] for size in t}
test_RMSE_svr_tscv = {size: [] for size in t}
train_RMSE_var = {size: [] for size in t}
test_RMSE_var = {size: [] for size in t}
tiempo_var = {size: [] for size in t}
tiempo_msvr = {size: [] for size in t}
tiempo_msvr_tscv = {size: [] for size in t}

for size in t:
      a=size
      for no in range(100):
        print(f"-------------------------Tamaño {a}-------------------------")
        print(f"-------------------------Iteration {no}--------------------------")

        # Read series
        series = pd.read_excel(f"XXX{size}.xlsx",sheet_name=f"Iteration_{no+1}")
        

        series = pd.DataFrame(
            serie.T,
            columns=["Y1", "Y2"]
        )

        # --------------------------------
        # Forecasting experiment
        # --------------------------------
        train_size = int(len(series) * 0.7)
        train, test = series.iloc[:train_size], series.iloc[train_size:]
        test = test.reset_index(drop=True)
        #Partial autocorrelation analysis
        pacf_var1 = pacf(train['Y1'], nlags=16)
        pacf_var2 = pacf(train['Y2'], nlags=16)
        banda= 1.96 / np.sqrt(size)
        rezago_elegido_1 = rezago_sig(pacf_var1, banda)
        rezago_elegido_2 = rezago_sig(pacf_var2, banda)
        enumerated_list = list(enumerate(pacf_var1))
        reversed_enumerated_list = list(reversed(enumerated_list))
        filtered_indices = [i for i, x in reversed_enumerated_list if abs(x) > banda]
        #rez= int(min(rezago_elegido_1, rezago_elegido_2))
        rez=p
        print(f"----------------Rezago {rez}--------------")
        # Fit the VAR model
        start_time = time.time()
        model_var = VAR(train)
        results_var = model_var.fit(maxlags=p)
        lag_order = results_var.k_ar
        modelo_var_train = []
        modelo_var_test = []
        #Generate in-sample predictions
        train_pred = results_var.fittedvalues
        # Generate out-of-sample forecasts
        test_pred=[]
        input_data = train.values[-rez:]
        for i in range(len(test)):
            pred = results_var.forecast(y=input_data, steps=h)
            test_pred.append(pred[0])
            input_data = np.vstack([input_data[1:], test.values[i:i+1]])
        test_pred = np.array(test_pred)
        train_rmse_var = np.sqrt(mean_squared_error(train.values[rez:], train_pred))
        test_rmse_var = np.sqrt(mean_squared_error(test.values, test_pred))
        train_RMSE_var[size].append(train_rmse_var)
        test_RMSE_var[size].append(test_rmse_var)
        print("Termine de ajustar modelo VAR")
        end_time = time.time()
        execution_time = end_time - start_time
        tiempo_var[size].append(execution_time)

        # Fit the MSVR model
        start_time = time.time()
        fechas = pd.DataFrame(list(range(len(series))))
        total = pd.concat([fechas,series], axis=1).values
        dim=len(total)
        #Dataset construction
        data=Base(total)
        data= data.base
        #Create the supervised learning dataset
        dataset = create_dataset_rez(data,dim,h,col,rez)
        X, Y = dataset[:, :(0 - h*2)], dataset[:, (0-h*2):]
        #Train-test split
        X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size=0.3, shuffle=False)
        #Feature standardization
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        scaler_X.fit(X_train)
        scaler_y.fit(y_train)
        X_train_nor = scaler_X.transform(X_train)
        X_test_nor = scaler_X.transform(X_test)
        y_train_nor = scaler_y.transform(y_train)
        y_test_nor = scaler_y.transform(y_test)
        pipe = Pipeline([
            ('MSVR', CustomMSVR(kernel='rbf', degree=3, gamma=0, coef0=0.0, tol=0.001, C=1.0, epsilon=0.1))
        ])
        hyperparameters = {
            #'MSVR__kernel': ['poly'],
            'MSVR__kernel': ['poly','rbf','linear'],
            'MSVR__degree': [2,5],
            #'MSVR__degree': [1],
            'MSVR__gamma': [0.5,1],
            'MSVR__coef0': [0.1,0.5,1],
            'MSVR__C': [5,9,11,13],
            'MSVR__epsilon':[1,2],
        }

        #Con tscv 
        tscv=TimeSeriesSplit(n_splits=5)
        bm_tscv = RandomizedSearchCV(pipe, hyperparameters, n_iter=15, scoring='neg_mean_squared_error', cv=tscv, verbose=0, error_score='raise')
        best_model_tscv = bm_tscv.fit(X_train_nor, y_train_nor)
        best_params_tscv = bm_tscv.best_params_
        msvr_tscv = MSVR(kernel=bm_tscv.best_params_.get("MSVR__kernel"), gamma=bm_tscv.best_params_.get("MSVR__gamma"),
                        epsilon=bm_tscv.best_params_.get("MSVR__epsilon"), C=bm_tscv.best_params_.get("MSVR__C"),
                        degree=bm_tscv.best_params_.get("MSVR__degree"), coef0=bm_tscv.best_params_.get("MSVR__coef0"), tol=0.01)
        msvr_tscv.fit(X_train_nor, y_train_nor)
        trainPred_svr_nor_tscv = msvr_tscv.predict(X_train_nor)
        testPred_svr_nor_tscv = msvr_tscv.predict(X_test_nor)
        trainPred_svr_tscv  = pd.DataFrame(scaler_y.inverse_transform(trainPred_svr_nor_tscv))
        testPred_svr_tscv  = pd.DataFrame(scaler_y.inverse_transform(testPred_svr_nor_tscv))
        train_rmse_svr_tscv = rmse(y_train, trainPred_svr_tscv)
        test_rmse_svr_tscv = rmse(y_test, testPred_svr_tscv)
        hiperparametros_svr_tscv[size].append(best_params_tscv)
        vectores_soporte_tscv[size].append(msvr_tscv.NSV)
        train_RMSE_svr_tscv[size].append(train_rmse_svr_tscv)
        test_RMSE_svr_tscv[size].append(test_rmse_svr_tscv)
        end_time = time.time()
        execution_time_tscv = end_time - start_time
        tiempo_msvr_tscv[size].append(end_time - start_time) 

        #Sin tscv 
        bm = RandomizedSearchCV(pipe, hyperparameters, n_iter=15, scoring='neg_mean_squared_error', cv=5, verbose=0, error_score='raise', random_state=42)
        best_model = bm.fit(X_train_nor, y_train_nor)
        best_params = bm.best_params_
        msvr = MSVR(kernel=bm.best_params_.get("MSVR__kernel"), gamma=bm.best_params_.get("MSVR__gamma"),
                        epsilon=bm.best_params_.get("MSVR__epsilon"), C=bm.best_params_.get("MSVR__C"),
                        degree=bm.best_params_.get("MSVR__degree"), coef0=bm.best_params_.get("MSVR__coef0"), tol=0.01)
        msvr.fit(X_train_nor, y_train_nor)
        trainPred_svr_nor = msvr.predict(X_train_nor)
        testPred_svr_nor = msvr.predict(X_test_nor)
        trainPred_svr  = pd.DataFrame(scaler_y.inverse_transform(trainPred_svr_nor))
        testPred_svr  = pd.DataFrame(scaler_y.inverse_transform(testPred_svr_nor))
        train_rmse_svr = rmse(y_train, trainPred_svr)
        #print(train_rmse_svr)
        test_rmse_svr = rmse(y_test, testPred_svr)
        hiperparametros_svr[size].append(best_params)
        vectores_soporte[size].append(msvr.NSV)
        train_RMSE_svr[size].append(train_rmse_svr)
        test_RMSE_svr[size].append(test_rmse_svr)
        end_time = time.time()
        execution_time = end_time - start_time
        tiempo_msvr[size].append(end_time - start_time)

filename = f'results_ VAR_Bivariate.csv'
with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Tamaño', 'Modelo', 'Hiperparametros', 'Vectores_soporte', 'Train RMSE', 'Test RMSE', 'Tiempo'])
        for size in t:
            writer.writerow([f'Tamaño: {size}', '', '', '', '', '', ''])
            for i in range(100):
                writer.writerow(['SVR', hiperparametros_svr[size][i], vectores_soporte[size][i], train_RMSE_svr[size][i], test_RMSE_svr[size][i], tiempo_msvr[size][i]])
            for i in range(100):
                writer.writerow(['SVR_tscv', hiperparametros_svr_tscv[size][i], vectores_soporte_tscv[size][i], train_RMSE_svr_tscv[size][i], test_RMSE_svr_tscv[size][i], 'N/A'])
            for i in range(100):  
                writer.writerow(['VAR ', 'N/A', 'N/A', train_RMSE_var[size][i], test_RMSE_var[size][i], tiempo_var[size][i]])
            

        print(f'Resultados guardados en {filename}')
        
 
 